# EDA: Detecção de Fraude (IEEE-CIS)

Semana 1: investigar estrutura dos dados, padrões de valores ausentes por classe, e definir o split temporal.

In [1]:
import sys
sys.path.append('..')

from src.data_loader import load_raw_data
from src.preprocessing import report_missing_by_class, temporal_train_test_split

df = load_raw_data()
df.shape

(590540, 434)

In [2]:
df_ordenado = df.sort_values(by="TransactionDT")
df_ordenado.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [9]:
valor_de_corte = df['TransactionDT'].quantile(0.7)

## Taxa de fraude e desbalanceamento

In [ ]:
df['isFraud'].value_counts(normalize=True)

## Valores ausentes: fraude vs. não-fraude

A ausência de um dado pode ser, ela mesma, um sinal — checar antes de imputar.

In [ ]:
report_missing_by_class(df).head(20)

## Split temporal

In [ ]:
train_df, test_df = temporal_train_test_split(df)
print(train_df.shape, test_df.shape)

In [ ]:
print(df.shape)
print(df.columns[:10].tolist())

In [3]:
print(df.shape)

(590540, 434)


In [4]:
import matplotlib.pyplot as plt
import numpy as np

COLOR_LEGIT = "#2a78d6"
COLOR_FRAUD = "#eb6834"
COLOR_MUTED = "#898781"
COLOR_GRID = "#e1e0d9"
COLOR_INK = "#0b0b0b"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": COLOR_GRID,
    "axes.labelcolor": COLOR_INK,
    "text.color": COLOR_INK,
    "xtick.color": COLOR_MUTED,
    "ytick.color": COLOR_MUTED,
    "grid.color": COLOR_GRID,
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

def style_ax(ax, title=None):
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, loc="left", fontsize=11, color=COLOR_INK)
    return ax

print("ok")

Matplotlib is building the font cache; this may take a moment.


ok


In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["TransactionAmt"], bins=80, color=COLOR_LEGIT, edgecolor="white", linewidth=0.3, zorder=3)
ax.set_xscale("log")
ax.set_xlabel("TransactionAmt (log)")
ax.set_ylabel("Contagem")
style_ax(ax, "Distribuição de TransactionAmt (todas as transações)")
plt.tight_layout()
plt.close(fig)
print("ok cell 1")

cat_cols = ["ProductCD", "card4", "card6", "DeviceType"]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts(dropna=False)
    ax.bar(counts.index.astype(str), counts.values, color=COLOR_LEGIT, zorder=3)
    style_ax(ax, col)
    ax.set_ylabel("Contagem")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.close(fig)
print("ok cell 2")

ok cell 1
ok cell 2


In [6]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.logspace(
    np.log10(df["TransactionAmt"].clip(lower=1).min()),
    np.log10(df["TransactionAmt"].max()),
    60,
)
ax.hist(df.loc[df["isFraud"] == 0, "TransactionAmt"], bins=bins, density=True,
        alpha=0.6, color=COLOR_LEGIT, label="Não-fraude", zorder=3)
ax.hist(df.loc[df["isFraud"] == 1, "TransactionAmt"], bins=bins, density=True,
        alpha=0.6, color=COLOR_FRAUD, label="Fraude", zorder=3)
ax.set_xscale("log")
ax.set_xlabel("TransactionAmt (log)")
ax.set_ylabel("Densidade")
style_ax(ax, "TransactionAmt por classe")
ax.legend(frameon=False)
plt.tight_layout()
plt.close(fig)
print("ok cell 3")

def fraud_rate_by(df, col):
    grouped = df.groupby(col, observed=True, dropna=False)["isFraud"].agg(["mean", "count"])
    return grouped.sort_values("mean", ascending=False)

overall_rate = df["isFraud"].mean()
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, cat_cols):
    rates = fraud_rate_by(df, col)
    ax.bar(rates.index.astype(str), rates["mean"], color=COLOR_LEGIT, zorder=3)
    ax.axhline(overall_rate, color=COLOR_MUTED, linestyle="--", linewidth=1, zorder=2,
               label=f"Média geral ({overall_rate:.2%})")
    style_ax(ax, f"Taxa de fraude por {col}")
    ax.set_ylabel("Taxa de fraude")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.close(fig)
print("ok cell 4")
print(fraud_rate_by(df, "ProductCD"))

ok cell 3
ok cell 4
               mean   count
ProductCD                  
C          0.116873   68519
S          0.058996   11628
H          0.047662   33024
R          0.037826   37699
W          0.020399  439670


In [13]:
from src.preprocessing import temporal_train_test_split, report_missing_by_class

train_df, test_df = temporal_train_test_split(df)
print(train_df.shape, test_df.shape)
print("train fraud rate:", train_df['isFraud'].mean())
print("test fraud rate:", test_df['isFraud'].mean())
print("train fraud count:", train_df['isFraud'].sum())
print("test fraud count:", test_df['isFraud'].sum())
print("overall fraud rate:", df['isFraud'].mean())
print("train time range (days):", (train_df['TransactionDT'].max()-train_df['TransactionDT'].min())/86400)
print("test time range (days):", (test_df['TransactionDT'].max()-test_df['TransactionDT'].min())/86400)
print("total time range (days):", (df['TransactionDT'].max()-df['TransactionDT'].min())/86400)

(472432, 434) (118108, 434)
train fraud rate: 0.03513521522674162
test fraud rate: 0.034409184813899145
train fraud count: 16599
test fraud count: 4064
overall fraud rate: 0.03499000914417313
train time range (days): 140.12085648148147
test time range (days): 41.87767361111111
total time range (days): 181.99920138888888


In [14]:
rep = report_missing_by_class(df)
print(rep.head(15).to_string())
print("---")
print("cols with diff > 0.3:", (rep['diff'] > 0.3).sum())
print("cols with diff > 0.5:", (rep['diff'] > 0.5).sum())

               missing_rate_fraud  missing_rate_legit      diff
R_emaildomain            0.456662            0.778787  0.322126
id_02                    0.456904            0.772495  0.315591
id_36                    0.456904            0.772296  0.315393
id_37                    0.456904            0.772296  0.315393
id_35                    0.456904            0.772296  0.315393
id_38                    0.456904            0.772296  0.315393
id_15                    0.456904            0.772296  0.315393
id_11                    0.457097            0.772302  0.315205
id_29                    0.457097            0.772302  0.315205
id_28                    0.457097            0.772302  0.315205
DeviceType               0.457436            0.772584  0.315148
id_12                    0.452258            0.766765  0.314508
id_01                    0.452258            0.766765  0.314508
id_31                    0.459372            0.773441  0.314069
id_05                    0.468519       